# 04 — Sentiment Analysis
Scores each ingredient mention with VADER, writes results to PostgreSQL, and exports a final CSV for Power BI.

In [ ]:
import os
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from sqlalchemy import create_engine
from dotenv import load_dotenv
from tqdm import tqdm

load_dotenv()
analyzer = SentimentIntensityAnalyzer()
df = pd.read_csv('../data/processed/ingredient_mentions_raw.csv')

In [ ]:
def score_sentiment(text: str) -> dict:
    scores = analyzer.polarity_scores(text)
    compound = scores['compound']
    if compound >= 0.05:
        label = 'positive'
    elif compound <= -0.05:
        label = 'negative'
    else:
        label = 'neutral'
    return {'sentiment_score': compound, 'sentiment_label': label}

tqdm.pandas(desc='Scoring sentiment')
sentiment = df['sentence'].progress_apply(score_sentiment)
df = df.join(pd.DataFrame(sentiment.tolist()))
df.head()

In [ ]:
# Quick summary
summary = (
    df.groupby('ingredient')
    .agg(
        mention_count=('sentiment_score', 'count'),
        avg_sentiment=('sentiment_score', 'mean'),
        negative_pct=('sentiment_label', lambda x: (x == 'negative').mean() * 100)
    )
    .round(3)
    .sort_values('avg_sentiment')
)
print(summary[summary['mention_count'] >= 10].head(15))

In [ ]:
# Save processed CSV for Power BI
df.to_csv('../data/processed/ingredient_sentiment_final.csv', index=False)

# Load to PostgreSQL
DB_URL = (
    f"postgresql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}"
    f"@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
)
engine = create_engine(DB_URL)
df.to_sql('ingredient_mentions', engine, if_exists='append', index=False)
print('Loaded to PostgreSQL ✓')